In [ ]:
import cv2
from pathlib import Path
import os
import numpy as np

class DocumentProcessor():
    # Valores iniciais do método init
    def __init__(self, mostrar=False, d=9, sigmaColor=20, sigmaSpace=20, blockSize=21, canny_threshold1=20, canny_threshold2=100):
        self.mostrar = mostrar
        self.d = d
        self.sigmaColor = sigmaColor
        self.sigmaSpace = sigmaSpace
        self.blockSize = blockSize
        self.canny_threshold1 = canny_threshold1
        self.canny_threshold2 = canny_threshold2

        self.valid_extensions = (".jpg", ".jpeg", ".png", ".bmp")

    #===========================================#
    #              Funções do PipeLine          #
    #===========================================#
    
    # mostrar imagem
    def _mostrar_img(self, img):
        cv2.imshow("amostra",img)
        cv2.waitKey()
        cv2.destroyAllWindows()

    # Leitura da Imagem
    def _read_image(self, path):
        img = cv2.imread(path)
        if img is not None:
            return img
        else:
            print("Imagem não carregada")
            return None

    # Filtro para Grayscale
    def _para_cinza(self, img):
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return img

    # Filtro para suavizar
    def _para_suavizar(self, img):
        #img = cv2.GaussianBlur(img, (5,5), 0 )
        #img = cv2.medianBlur(img, 5)
        img = cv2.bilateralFilter(img, self.d, self.sigmaColor, self.sigmaSpace)
        return img

    # Threshold
    def _para_threshold(self, img):
        img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, self.blockSize, C=10)
        return img

    def _aplicar_otsu(self, img):
        limiar, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        print(f"Limiar encontrado pelo Otsu: {limiar}")
        return img

    # Canny
    def _aplicar_canny(self, img):
        img = cv2.Canny(img, self.canny_threshold1, self.canny_threshold2)
        return img

    # Morfologia
    def _aplicar_morfologia(self, img):
        kernel = np.ones((5,5), np.uint8)
        img = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)
        return img

    # Resize para 256x256
    def _aplicar_resize(self, img):
        img = cv2.resize(img, (256,256), interpolation=cv2.INTER_AREA)
        #print(img.shape)
        return img
    
    # Salva a imagem na pasta configurada
    def _salvar_imagem(self, img, path):
        try:
            cv2.imwrite(path, img)
            return True
        except Exception as e:
            print(f"Erro ao salvar no disco: {e}")
            return False
        
    #===========================================#
    #              Pipeline da imagem           #
    #===========================================#

    def preprocess_img(self, input_path, output_path, mostrar=False):
        img = self._read_image(input_path)
        if img is None: 
            return False

        img = self._para_cinza(img)
        img = self._para_suavizar(img)
        img = self._para_threshold(img)
        #img = self._aplicar_otsu(img)
        img = self._aplicar_canny(img)
        img = self._aplicar_morfologia(img)
        img = self._aplicar_resize(img)

        if mostrar:
            self._mostrar_img(img)

        return self._salvar_imagem(img, output_path)

    #===========================================#
    #              Processo em Lote             #
    #===========================================#

    def process_batch(self, input_dir, output_dir, limite=None):
        '''Processamento em Lote'''
        try:
            #Criar diretório se não existir
            Path(output_dir).mkdir(parents=True, exist_ok=True) 

            #Criar lista de caminhos e tratar minusculo e validar a extensao
            files = [f for f in os.listdir(input_dir) if f.lower().endswith(self.valid_extensions)]
            # Controle de imagens por lote
            files = files[:limite]
            
            if not files:
                print(f"Não há imagens na pasta {input_dir}")
                return

            print(f"Iniciando pré processamento de {len(files)} Imagens....")
            sucess = 0
            for filename in files:
                in_path = os.path.join(input_dir, filename)
                # Quebrar extensao, pega o nome do arquivo sem a extensao
                base_name = os.path.splitext(filename)[0]
                out_path = os.path.join(output_dir, f'{base_name}.png')
                if self.preprocess_img(in_path, out_path, mostrar=self.mostrar):  # usar depoi fase 6: self.preprocess_img(in_path, out_path, mostrar=False) 
                    sucess+=1
            print(f"{sucess} Imagens processadas com sucesso")
        except Exception as e:
            print(f"Erro ao processar em Lote: {e}")



In [ ]:
# Instanciar a classe
# Parâmetros configuráveis: mostrar=False, d=9, sigmaColor=20, sigmaSpace=20, blockSize=21, canny_threshold1=20, canny_threshold2=100
processor = DocumentProcessor(mostrar=False)

#processor.preprocess_img('raw_images/def_front/cast_def_0_2.jpeg', mostrar=True)

# Chama o método para execução do PipeLine, Configurar limite para None
processor.process_batch('data/raw', 'data/processed_images', limite=5)

Iniciando pré processamento de 5 Imagens....
5 Imagens processadas com sucesso
